# Türkçe Metinlerden Duygu Sınıflandırma

Bu proje, Türkçe bir metnin **Mutluluk, Üzüntü, Öfke, Korku, Nötr veya Utanç** sınıflarından hangisine ait olduğunu tahmin eder. Notebook, proje yönergesinin yedi aşamasına göre düzenlenmiştir.

**Çalıştırma:** Colab'da `Çalışma zamanı → Tümünü çalıştır` seçeneğini kullanın. Veri dosyası varsayılan olarak `/content/drive/MyDrive/colab_data/train_set_emotions.csv` konumundan okunur. Dosya oradaysa tekrar yüklemeniz gerekmez. En son hücre cümle girmenizi bekler.

**Doğrulama durumu:** Bu düzenleme sırasında özgün CSV sağlanmadığından gerçek veriyle yeniden eğitim yapılmamıştır. Eski çıktılar, yeni kodla karışmaması için kaldırılmıştır. Sayısal sonuçlar ve rapor taslağı çalıştırınca üretilir. Veri kaynağı bilgisi aşağıda tamamlanmalıdır. Ayrı sunum ve bir sayfalık rapor dosyaları bu notebook'un yerine geçmez.

## 1. Problem Tanımı ve Proje Planı

| Alan | Açıklama |
|---|---|
| Proje adı | Türkçe Metinlerden Duygu Sınıflandırma |
| Problem cümlesi | Türkçe metin ve metinden türetilen uzunluk bilgileri kullanılarak duygu sınıfı tahmin edilebilir mi? |
| Girdiler X | Metin, kelime sayısı ve karakter sayısı |
| Hedef y | `label` sütunundaki duygu sınıfı |
| Problem türü | Denetimli öğrenme, çok sınıflı sınıflandırma |
| Kullanıcılar | Türkçe geri bildirimleri inceleyen araştırmacılar ve uygulama geliştiricileri |

**Gerekçe:** Çok sayıdaki metni tek tek okumak zaman alır. Metinleri duygu sınıflarına ayırmak, geri bildirimleri topluca incelemeyi kolaylaştırabilir. Bu çalışma bir eğitim prototipidir; kişinin gerçek psikolojik durumunu ölçmez ve insan değerlendirmesinin yerine geçmez.

**Plan:** Veriyi kontrol et → temizle → keşfet → eğitim/test ayır → eğitim içinde çapraz doğrulamayla modelleri karşılaştır → seçilen modeli test et → örnek tahmin ve kullanıcı uygulaması oluştur → sonuçları raporla.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

SEED = 42
plt.rcParams.update({"figure.figsize": (9, 5), "axes.titlesize": 13})
CIKTI = Path("emotion_proje_ciktilari")
CIKTI.mkdir(exist_ok=True)

## 2. Veri Toplama ve Hazırlama

**Veri kaynağı:** Kullanıcının sağladığı `train_set_emotions.csv`. Kaynağın URL'si, oluşturan kişi/kurum, etiketleme yöntemi ve kullanım izni bu dosyadan doğrulanamıyor. Aşağıdaki alanları teslimden önce gerçek bilgilerle doldurun; veri yapay olarak oluşturulduysa açıkça belirtin.

Orijinal notebook'ta 2.698 ham, 2.665 temiz kayıt görülmüştür. Bunlar önceki çalıştırmanın bilgileridir; aşağıdaki kod mevcut CSV'nin gerçek sayılarını hesaplar.

In [ ]:
VERI_KAYNAGI = "Belirtilmedi — gerçek kaynak veya oluşturma yöntemi eklenmeli"
ETIKETLEME_YONTEMI = "Belirtilmedi — etiketlerin nasıl üretildiği eklenmeli"
CSV_YOLU = Path("/content/drive/MyDrive/colab_data/train_set_emotions.csv")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

if not CSV_YOLU.exists():
    yerel = Path("train_set_emotions.csv")
    if yerel.exists():
        CSV_YOLU = yerel
    elif "google.colab" in sys.modules:
        from google.colab import files
        print("Drive'da CSV bulunamadı. train_set_emotions.csv dosyasını seçin.")
        yuklenen = files.upload()
        csvler = [ad for ad in yuklenen if ad.lower().endswith(".csv")]
        if len(csvler) != 1:
            raise ValueError("Lütfen yalnızca bir CSV dosyası yükleyin.")
        CSV_YOLU = Path(csvler[0])
    else:
        raise FileNotFoundError("CSV_YOLU değişkenini kendi dosyanızın yoluyla değiştirin.")

df_ham = pd.read_csv(CSV_YOLU)
if not {"text", "label"}.issubset(df_ham.columns):
    raise ValueError("CSV, text ve label sütunlarını içermelidir.")
print(f"Ham veri boyutu: {df_ham.shape}")
display(df_ham.head(10))
display(df_ham.isna().sum().rename("Eksik değer sayısı").to_frame())

### Veri sözlüğü

| Sütun | Tip | Anlamı ve rolü |
|---|---|---|
| id | Kimlik/metin | Kayıt kimliği; modele verilmez |
| text | Metin | Duygunun tahmin edileceği içerik; özellik |
| label | Kategorik | Duygu adı; hedef |
| label_id | Kategorik kod | Hedefin sayısal karşılığı; modele verilmez |
| kelime_sayisi | Sayısal | Metindeki boşlukla ayrılmış kelime sayısı; türetilmiş özellik |
| karakter_sayisi | Sayısal | Metnin karakter sayısı; türetilmiş özellik |

Son iki sütun metinden türetilir; bağımsız bir veri kaynağı değildir. Metinle birlikte modele dahil edilir ve temiz CSV'ye kaydedilir. Hedefi doğrudan açığa çıkaran `label_id` özellik olarak kullanılmaz.

In [ ]:
# Boş metin/hedefleri çıkar; boşlukları düzenle; tekrarları ve çelişkileri kontrol et.
df = df_ham.copy()
ham_sayi = len(df)
df = df.dropna(subset=["text", "label"]).copy()
df["text"] = df["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df["label"] = df["label"].astype(str).str.strip()
df = df[(df["text"] != "") & (df["label"] != "")].copy()
bos_silinen = ham_sayi - len(df)
beklenen = {"Mutluluk", "Üzüntü", "Öfke", "Korku", "Nötr", "Utanç"}
bilinmeyen = set(df["label"]) - beklenen
if bilinmeyen:
    raise ValueError(f"Etiketleri kontrol edin; beklenmeyen sınıflar: {bilinmeyen}")

# Aynı metne farklı etiket verilmişse rastgele bir etiketi seçmek yerine inceleme iste.
celiskiler = df.groupby("text")["label"].nunique()
celiskiler = celiskiler[celiskiler > 1]
if len(celiskiler):
    display(df[df["text"].isin(celiskiler.index)].sort_values("text"))
    raise ValueError("Aynı metin farklı etiketler taşıyor; bu kayıtları inceleyip düzeltin.")
tekrar_sayisi = int(df.duplicated(subset="text").sum())
df = df.drop_duplicates(subset="text").reset_index(drop=True)
if len(df) < 20 or set(df["label"]) != beklenen:
    raise ValueError("En az 20 kayıt ve beklenen altı duygu sınıfı bulunmalıdır.")

def ozellik_uret(metinler):
    metinler = pd.Series(metinler, dtype="string").str.replace(r"\s+", " ", regex=True).str.strip()
    return pd.DataFrame({"text": metinler,
                         "kelime_sayisi": metinler.str.split().str.len().astype(int),
                         "karakter_sayisi": metinler.str.len().astype(int)})

ozellikler = ozellik_uret(df["text"])
df["kelime_sayisi"] = ozellikler["kelime_sayisi"]
df["karakter_sayisi"] = ozellikler["karakter_sayisi"]
# Sayısal etiket varsa, temiz hedefle tutarlı bir eşleme oluştur.
etiket_kodlari = {"Mutluluk": 0, "Üzüntü": 1, "Öfke": 2, "Korku": 3, "Nötr": 4, "Utanç": 5}
df["label_id"] = df["label"].map(etiket_kodlari)
df.to_csv(CIKTI / "train_set_emotions_processed.csv", index=False, encoding="utf-8-sig")
print(f"Ham: {ham_sayi} | Boş kayıt çıkarılan: {bos_silinen} | Tekrar çıkarılan: {tekrar_sayisi} | Temiz: {len(df)}")
print(f"Kalan metin tekrarı: {df['text'].duplicated().sum()}")
display(df.dtypes.rename("Veri tipi").to_frame())

**Temizleme açıklaması:** Eksik veya boş metin ve hedefler çıkarılır; gereksiz boşluklar düzenlenir. Aynı metnin farklı etiketleri varsa işlem durdurularak inceleme istenir. Aynı metnin tekrarları eğitim/test ayrımından önce kaldırılır. Noktalama ve Türkçe karakterler korunur. `label_id` hedefle tutarlı biçimde oluşturulur. Kelime ve karakter sayıları eklenir. Benzer ama birebir aynı olmayan cümleler bu kontrolle tamamen tespit edilemez.

## 3. Keşifçi Veri Analizi ve Görselleştirme

Önce özet istatistikler, ardından sınıf dağılımı ve duyguya göre metin uzunluğu incelenir. Bu keşifler veri tanımak içindir; model seçimi test puanlarına göre yapılmaz.

In [ ]:
print("Temiz veri boyutu:", df.shape)
display(df.describe(include="all"))
sinif_sayilari = df["label"].value_counts().sort_index()
display(sinif_sayilari.rename("Kayıt sayısı").to_frame())
display(df.groupby("label")[["kelime_sayisi", "karakter_sayisi"]].agg(["mean", "median", "min", "max"]).round(2))
display(df.groupby("label")["label_id"].unique().rename("Etiket kodu").to_frame())

fig, ax = plt.subplots()
sinif_sayilari.plot.bar(ax=ax, color="#5978b8")
ax.set(title="Temiz veride duygu dağılımı", xlabel="Duygu sınıfı", ylabel="Metin sayısı")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(CIKTI / "01_duygu_dagilimi.png", dpi=160)
plt.show()

fig, ax = plt.subplots()
siniflar = sorted(df["label"].unique())
ax.boxplot([df.loc[df["label"] == s, "kelime_sayisi"] for s in siniflar])
ax.set_xticks(range(1, len(siniflar) + 1), siniflar, rotation=25)
ax.set(title="Duyguya göre metin uzunluğu", xlabel="Duygu sınıfı", ylabel="Kelime sayısı")
fig.tight_layout()
fig.savefig(CIKTI / "02_metin_uzunlugu.png", dpi=160)
plt.show()

ortalamalar = df.groupby("label")["kelime_sayisi"].mean()
display(Markdown(f"""### Keşif Bulguları
- Temizlik sonrası **{len(df)} metin ve {df['label'].nunique()} duygu sınıfı** bulunmaktadır.
- En kalabalık sınıf **{sinif_sayilari.idxmax()} ({sinif_sayilari.max()})**, en küçük sınıf **{sinif_sayilari.idxmin()} ({sinif_sayilari.min()})**; en büyük/en küçük oranı **{sinif_sayilari.max()/sinif_sayilari.min():.2f}**. Sınıf dengesi eşit model başarısını garanti etmez.
- Metin başına ortalama **{df['kelime_sayisi'].mean():.1f} kelime**, medyan **{df['kelime_sayisi'].median():.1f} kelime** vardır.
- Ortalama metin uzunluğu en yüksek sınıf **{ortalamalar.idxmax()} ({ortalamalar.max():.1f} kelime)**, en düşük sınıf **{ortalamalar.idxmin()} ({ortalamalar.min():.1f} kelime)**. Bu fark tek başına uzunluğun belirleyici olduğunu kanıtlamaz.
- **{tekrar_sayisi} tekrar kayıt** çıkarılmıştır. Benzer kalıplı cümleler kalabilir; bu durum genelleme başarısını olduğundan yüksek gösterebilir.
"""))

## 4. Model Kurma ve Eğitme

Verinin %80'i eğitim, %20'si son test için ayrılır. Sınıf oranlarını korumak için `stratify` kullanılır. TF-IDF ve sayısal ölçekleme Pipeline içinde öğrenildiği için test verisi dönüşümlerin eğitimine katılmaz.

**Model seçim gerekçesi:** Logistic Regression ve Linear SVM, TF-IDF gibi çok boyutlu metin özellikleriyle karşılaştırılabilecek basit sınıflandırıcılardır. Karakter n-gramları kelime parçalarını kullanarak bazı ek ve yazım farklılıklarını yakalayabilir; bu bir kök bulma algoritması değildir. Yönergede adı geçen DecisionTreeClassifier da karşılaştırmaya eklenmiştir; ağaç derinliği ve yaprak boyutu sınırlandırılmıştır.

Modeller, yalnızca eğitim verisinde 3 katlı çapraz doğrulamanın **makro F1** ortalamasına göre seçilir. Son test, seçilen model için bir kez raporlanır. Önceki notebook'ta aynı veri üzerinde denemeler yapıldığı için bu test tamamen yeni, dış bir veri değerlendirmesi sayılmaz.

In [ ]:
X = df[["text", "kelime_sayisi", "karakter_sayisi"]]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
if y_train.value_counts().min() < 3:
    raise ValueError("3 katlı çapraz doğrulama için her sınıfta en az 3 eğitim örneği gereklidir.")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

def model_olustur(siniflandirici, karakter=False):
    tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True) if karakter else TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    donusum = ColumnTransformer([
        ("metin", tfidf, "text"),
        ("uzunluk", MaxAbsScaler(), ["kelime_sayisi", "karakter_sayisi"])
    ])
    return Pipeline([("ozellikler", donusum), ("siniflandirici", siniflandirici)])

modeller = {
    "Kelime TF-IDF + Logistic Regression": model_olustur(LogisticRegression(max_iter=2000, random_state=SEED)),
    "Kelime TF-IDF + Linear SVM": model_olustur(LinearSVC(max_iter=10000, random_state=SEED)),
    "Karakter TF-IDF + Linear SVM": model_olustur(LinearSVC(max_iter=10000, random_state=SEED), karakter=True),
    "Kelime TF-IDF + Decision Tree": model_olustur(DecisionTreeClassifier(max_depth=20, min_samples_leaf=3, random_state=SEED))
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
karsilastirma = []
for ad, aday in modeller.items():
    sonuc = cross_validate(aday, X_train, y_train, cv=cv,
                           scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"}, n_jobs=1)
    karsilastirma.append({"Model": ad,
                         "CV doğruluk": sonuc["test_accuracy"].mean(),
                         "CV makro F1": sonuc["test_f1_macro"].mean(),
                         "CV F1 std": sonuc["test_f1_macro"].std()})
karsilastirma = pd.DataFrame(karsilastirma).sort_values("CV makro F1", ascending=False).reset_index(drop=True)
display(karsilastirma.round(4))
karsilastirma.to_csv(CIKTI / "model_karsilastirma.csv", index=False, encoding="utf-8-sig")
secilen_ad = karsilastirma.loc[0, "Model"]
model = modeller[secilen_ad]
model.fit(X_train, y_train)
print(f"Çapraz doğrulamayla seçilen model: {secilen_ad}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(karsilastirma["Model"], karsilastirma["CV makro F1"], xerr=karsilastirma["CV F1 std"], color="#5978b8")
ax.set(xlim=(0, 1), xlabel="3 katlı doğrulama makro F1 ortalaması", ylabel="Model", title="Eğitim verisinde model karşılaştırması")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(CIKTI / "03_model_karsilastirma.png", dpi=160)
plt.show()

## 5. Tahmin ve Değerlendirme

Doğruluk tüm doğru tahminlerin oranıdır. Makro F1 her sınıfa eşit ağırlık verir. Kesinlik, bir sınıfa yapılan tahminlerin ne kadarının doğru olduğunu; duyarlılık o sınıftaki gerçek örneklerin ne kadarının bulunduğunu gösterir.

In [ ]:
tahminler = model.predict(X_test)
dogruluk = accuracy_score(y_test, tahminler)
makro_f1 = f1_score(y_test, tahminler, average="macro")
egitim_dogruluk = accuracy_score(y_train, model.predict(X_train))
rapor = pd.DataFrame(classification_report(y_test, tahminler, output_dict=True, zero_division=0)).T
print(f"Seçilen model: {secilen_ad}")
print(f"Eğitim doğruluğu: %{egitim_dogruluk*100:.2f}")
print(f"Test doğruluğu: %{dogruluk*100:.2f} | Test makro F1: {makro_f1:.4f}")
display(rapor.round(3))
rapor.to_csv(CIKTI / "siniflandirma_raporu.csv", encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(y_test, tahminler, labels=model.classes_, cmap="Blues", ax=ax, colorbar=False)
ax.set(title=f"{secilen_ad}\nTest doğruluğu: %{dogruluk*100:.2f}", xlabel="Tahmin edilen duygu", ylabel="Gerçek duygu")
fig.tight_layout()
fig.savefig(CIKTI / "04_karisiklik_matrisi.png", dpi=180)
plt.show()

sinif_f1 = rapor.loc[model.classes_, "f1-score"]
display(Markdown(f"""### Değerlendirme Notu
Seçilen model, **{len(y_test)} test örneğinde %{dogruluk*100:.2f} doğruluk** ve **{makro_f1:.3f} makro F1** elde etti. Sınıf bazında F1 değeri en düşük sınıf **{sinif_f1.idxmin()} ({sinif_f1.min():.3f})** oldu. Eğitim ve test doğruluğu farkı **{(egitim_dogruluk-dogruluk)*100:.2f} yüzde puan**. Eğitim başarısının daha yüksek olması aşırı öğrenme açısından incelenmelidir; bu fark tek başına aşırı öğrenmeyi kanıtlamaz.

Bu sonuçlar aynı veri kaynağındaki örnekler için bir ölçümdür. Bağımsız kaynaklardan gelen metinlerle doğrulanmadan modelin genel olarak güvenilir olduğu söylenemez. Alay, olumsuzluk, karma duygular, yazım hataları ve çok kısa ifadeler hata yaratabilir. Altı sınıf dışında kalan duygular da zorunlu olarak bu sınıflardan birine atanır.

**İyileştirmeler:** Veri kaynağı ve etiket kalitesi doğrulanmalı; yakın anlamlı tekrarlar incelenmeli; doğal ve çeşitli yeni metinler eklenmeli; bağımsız bir dış test setiyle değerlendirme yapılmalıdır. Karakter parçaları ek farklılıklarını kısmen yakalasa da bağlamı veya Türkçe dil bilgisini doğrudan anlamaz.
"""))

In [ ]:
# Bu örneklerin beklenen sınıfları insan yorumudur; modelin başarısını garanti etmez.
ornekler = pd.DataFrame({
    "Girdi": ["Kazandığımı öğrenince sevinçten yerimde duramadım.",
              "Onun gidişinden sonra içimde büyük bir boşluk kaldı.",
              "Karanlık sokaktaki ayak sesleri beni ürküttü.",
              "Bana tekrar yalan söylemene çok sinirlendim.",
              "Dosyayı masanın üzerindeki klasöre yerleştirdim.",
              "Herkesin önünde yanlış kişiye sarılınca yüzüm kızardı."],
    "Beklenen duygu": ["Mutluluk", "Üzüntü", "Korku", "Öfke", "Nötr", "Utanç"]
})
# Veri setinde bulunan cümleleri yeni örnek olarak sunma.
ornekler = ornekler[~ornekler["Girdi"].isin(set(df["text"]))].reset_index(drop=True)
if len(ornekler) < 3:
    raise ValueError("En az 3 örnek veri setinden farklı olmalıdır; yeni cümleler yazın.")
ornekler["Tahmin"] = model.predict(ozellik_uret(ornekler["Girdi"]))
ornekler["Yorum"] = np.where(ornekler["Tahmin"] == ornekler["Beklenen duygu"],
                              "Beklenen duyguyla uyumlu.", "Beklenen duygudan farklı; bağlam ve etiket yorumu incelenmeli.")
display(ornekler)
ornekler.to_csv(CIKTI / "yeni_ornek_tahminleri.csv", index=False, encoding="utf-8-sig")
print("Bu küçük örnek tablosu, bağımsız test kümesinin yerine geçmez.")

## 6. Akıllı Tahmin Aracı

Fonksiyon kullanıcıdan bir cümle alır, eğitimdekiyle aynı özellikleri üretir ve tahmini anlamlı bir cümleyle verir. Boş giriş kabul edilmez. Örnek çalıştırma çıktısını notebook'ta saklayın.

In [ ]:
def tahmin_et(cumle=None):
    if cumle is None:
        cumle = input("Bir cümle yaz: ")
    if not isinstance(cumle, str) or not cumle.strip():
        print("Lütfen boş olmayan bir cümle girin.")
        return None
    sonuc = model.predict(ozellik_uret([cumle]))[0]
    print(f'“{cumle.strip()}” cümlesinin tahmin edilen duygusu: {sonuc}.')
    return sonuc

# Kayıtlı örnek çalıştırma; etkileşimli kullanım son hücrededir.
_ = tahmin_et("Bugün çok mutluyum!")

## 7. Dokümantasyon ve Teslim

Aşağıdaki kısa rapor taslağı çalıştırıldığında gerçek sonuçlarla doldurulur. Bir sayfalık rapor dosyasına aktarılmadan önce kaynak bilgisi ve yazar bilgileri tamamlanmalıdır. Sayfa sayısı, Word/PDF düzeninde ayrıca kontrol edilmelidir.

**3–5 slaytlık sunum için 5 slayt planı:**
1. Problem, amaç, hedef kullanıcılar ve altı duygu sınıfı.
2. Veri kaynağı, ham/temiz kayıt sayıları, özellikler ve temizleme. Sınıf dağılımı grafiği.
3. TF-IDF, sayısal özellikler, eğitim/test ayrımı ve karşılaştırılan modeller.
4. Çapraz doğrulamayla seçilen model, doğruluk, makro F1 ve **karışıklık matrisi**.
5. Yeni cümle örnekleri, mini uygulama, sınırlılıklar ve iyileştirmeler.

**Teslim kontrolü:** Kaynak bilgilerini tamamla; bütün hücreleri çalıştır; hata olmadığını kontrol et; çıktılarıyla notebook'u kaydet; temiz CSV'yi, 1 sayfalık raporu ve 3–5 slaytlık sunumu ekle. Proje Tanım Formu, veri sözlüğü ve değerlendirme notu bu notebook içinde yer alır. Çalıştırma sırasında üretilen grafikler ve tablolar `emotion_proje_ciktilari` klasöründedir. Bunları indirerek saklayın.

In [ ]:
kisa_rapor = f"""# Türkçe Metinlerden Duygu Sınıflandırma

**Problem ve amaç:** Türkçe metinleri Mutluluk, Üzüntü, Öfke, Korku, Nötr ve Utanç sınıflarına ayırarak metin incelemesini kolaylaştıran bir eğitim prototipi geliştirilmiştir.

**Veri:** Kaynak: {VERI_KAYNAGI}. Etiketleme: {ETIKETLEME_YONTEMI}. {ham_sayi} ham kayıttan {bos_silinen} boş/geçersiz metin-hedef kaydı ve {tekrar_sayisi} tekrar çıkarılmış, {len(df)} kayıt kalmıştır. Metin, kelime sayısı ve karakter sayısı özellik; duygu etiketi hedef olarak kullanılmıştır. Kimlik ve hedef kodu model girdisinden çıkarılmıştır.

**Yöntem:** Veri {len(X_train)} eğitim ve {len(X_test)} test örneğine ayrılmıştır. Kelime/karakter TF-IDF ve ölçeklenmiş uzunluk özellikleri Pipeline içinde işlenmiştir. Logistic Regression, Linear SVM ve Decision Tree seçenekleri eğitim verisinde 3 katlı çapraz doğrulamayla karşılaştırılmıştır. Makro F1 ortalamasına göre {secilen_ad} seçilmiştir.

**Sonuç:** Seçilen modelin test doğruluğu %{dogruluk*100:.2f}, makro F1 değeri {makro_f1:.3f} olmuştur. En düşük sınıf F1 değeri {sinif_f1.idxmin()} için {sinif_f1.min():.3f} bulunmuştur. Veri setinde birebir bulunmayan {len(ornekler)} yeni cümle için tahmin tablosu ve kullanıcıdan girdi alan bir tahmin fonksiyonu hazırlanmıştır.

**Çıkarım ve sınırlılıklar:** Sonuçlar bu veri kaynağı ve bölünmesiyle sınırlıdır. Veri kaynağı/etiket kalitesi, benzer cümleler ve önceki deneyler değerlendirmeyi etkileyebilir. Model karma duyguları, alayı ve altı sınıf dışındaki ifadeleri güvenilir biçimde ayırt etmeyebilir. Daha çeşitli veri ve bağımsız dış test gerekir; uygulama psikolojik değerlendirme aracı değildir.
"""
display(Markdown(kisa_rapor))
(CIKTI / "kisa_proje_raporu_taslagi.md").write_text(kisa_rapor, encoding="utf-8")
print(f"Grafik, tablo, temiz CSV ve rapor taslağı klasörü: {CIKTI.resolve()}")

In [ ]:
# Son hücre: kullanıcıdan girdi alır. Yazıp Enter'a basın.
_ = tahmin_et()